# 🏦 03. Track FOMC: FOMC-RoBERTa & Probability Calibration
### Financial News Sentiment & Market Association System (ARCAS)

> **Paper Reference:** Shah, A., Paturi, S., & Chava, S. (2023). *"Trillion Dollar Words: A New Financial Dataset, Task & Market Analysis on Federal Reserve Communications"*, ACL 2023.
> **Target Assets:** `RATE_UST` (US 10-Year Treasury Yield) & `SPX` (S&P 500 Index)
> **Checkpoint:** `gtfintechlab/FOMC-RoBERTa` (Pre-trained Domain Expert)
> **Baseline:** TF-IDF (1-2 gram) + Logistic Regression (M1)
> **Architecture:** Dual-Compatible (Local CPU & Kaggle GPU Auto-Detect)

---
### Metodologi & Batasan Anti-Leakage:
1. **Holdout Terkunci:** 495 sampel test holdout (`test-5768.xlsx`) bersifat *strictly unseen* dan baru dibuka di Bagian 6.
2. **Tuning Suhu (M6):** Parameter temperatur $T_{fomc}$ dioptimalkan murni pada 194 sampel validasi (10% stratified split).
3. **Dual Transmission:** Satu sikap moneter ditransmisikan berlawanan arah (`RATE_UST` vs `SPX`).

In [1]:
from IPython.display import display

# =============================================================================
# 1. SETUP & AUTO-DETECT ENVIRONMENT (OPSI B: DUAL-COMPATIBLE)
# =============================================================================
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import numpy as np
import torch

# Deteksi apakah notebook berjalan di Kaggle cloud atau lokal
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()

if IS_KAGGLE:
    print("🚀 Environment: Cloud Kaggle (GPU Mode)")
    if not Path("ARCAS").exists():
        !git clone https://github.com/Josshua-DSA/ARCAS.git
    %cd ARCAS
    ROOT_PATH = Path("/kaggle/working/ARCAS").resolve()
else:
    print("💻 Environment: Local Workspace (CPU Mode)")
    ROOT_PATH = Path(".").resolve()

if str(ROOT_PATH) not in sys.path:
    sys.path.insert(0, str(ROOT_PATH))

# Hardware & Seed Locking
SEED = 5768
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target Device : {DEVICE}")
print(f"Project Root  : {ROOT_PATH}")

ModuleNotFoundError: No module named 'torch'

---
## 2. Pemuatan Data & Partisi Stratifikasi Anti-Leakage
Memuat varian terkunci `lab-manual-split-combine-train-5768.xlsx` dan membaginya menjadi 90% latih (1.745 sampel) dan 10% validasi (194 sampel).

In [ ]:
from model.fomc.src.data_loader import (
    LABEL_TO_STANCE,
    load_fomc_test_holdout,
    load_fomc_train_val_split,
)

# 1. Load stratified train & validation split
df_train, df_val = load_fomc_train_val_split(val_ratio=0.10, seed=SEED)

# 2. Load frozen holdout test set (read-only)
df_test = load_fomc_test_holdout()

print(f"Data Latih Baseline (90%) : {len(df_train)} baris")
print(f"Data Validasi Suhu  (10%) : {len(df_val)} baris")
print(f"Test Holdout Beku         : {len(df_test)} baris")

display(df_train.head(3))

In [ ]:
import matplotlib.pyplot as plt

# Visualisasi distribusi kelas per partisi data
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
splits = [("Train (1.745)", df_train), ("Validation (194)", df_val), ("Holdout (495)", df_test)]

for ax, (title, df_subset) in zip(axes, splits):
    counts = df_subset["stance"].value_counts(normalize=True) * 100
    counts.plot(kind="bar", ax=ax, color=["#4A90E2", "#50E3C2", "#F5A623"])
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Persentase (%)")
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    for i, v in enumerate(counts):
        ax.text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")

plt.suptitle("Distribusi Sikap Moneter (Hawkish/Dovish/Neutral) Antar Split Data", y=1.05, fontsize=14)
plt.show()

---
## 3. Pelatihan Model Baseline M1 (TF-IDF + Logistic Regression)
Melatih baseline n-gram pada 1.745 data latih dan mengekspor bobot ke `model/fomc/weights/baseline_fomc.joblib`.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

from model.fomc.src.model import train_fomc_baseline

# Latih baseline pada data train
vec, clf, b_path = train_fomc_baseline(df_train)
print(f"✅ Baseline berhasil dilatih dan disimpan ke: {b_path}")

# Evaluasi pada validation split
X_val = vec.transform(df_val["sentence"])
y_val_pred = clf.predict(X_val)

acc_base = accuracy_score(df_val["label"], y_val_pred)
f1_base = f1_score(df_val["label"], y_val_pred, average="macro")
print(f"Validation Accuracy Baseline : {acc_base:.4f}")
print(f"Validation Macro F1 Baseline : {f1_base:.4f}")

---
## 4. Ekstraksi Logits `FOMC-RoBERTa` (Shah et al., ACL 2023)
Memuat checkpoint pre-trained domain expert dan mengekstrak tensor logits mentah pada 194 kalimat validasi.

In [ ]:
from model.fomc.src.model import FOMCRoBERTaSentiment

print("Memuat model FOMC-RoBERTa...")
roberta = FOMCRoBERTaSentiment(device=str(DEVICE), lazy_load=False)

# Ekstraksi raw logits pada set validasi
val_texts = df_val["sentence"].tolist()
val_labels = df_val["label"].tolist()
val_logits = roberta.predict_raw_logits(val_texts)

print(f"Logits shape: {val_logits.shape} (N={len(val_texts)}, C=3)")
print("Contoh 3 logits pertama:")
print(val_logits[:3])

---
## 5. Kalibrasi Probabilitas via *Temperature Scaling* (M6)
Mencari skalar temperatur optimal $T_{fomc}$ dengan meminimalkan Negative Log-Likelihood (NLL) agar probabilitas tidak *overconfident* dan $ECE \le 0.10$.

In [ ]:
from model.fomc.src.calibrator import TemperatureScaler

scaler = TemperatureScaler()
opt_t = scaler.fit(val_logits, val_labels)

print(f"Temperatur Optimal (T)   : {opt_t:.4f}")
print(f"ECE Sebelum Kalibrasi     : {scaler.ece_pre:.4f}")
print(f"ECE Sesudah Kalibrasi     : {scaler.ece_post:.4f} (Target <= 0.10: {bool(scaler.ece_post is not None and scaler.ece_post <= 0.10)})")

# Simpan parameter suhu ke disk
temp_file = scaler.save()
print(f"✅ Parameter suhu tersimpan di: {temp_file}")

In [ ]:
from model.fomc.src.calibrator import softmax

# Visualisasi Reliability Diagram
probs_pre = softmax(val_logits, temperature=1.0)
probs_post = softmax(val_logits, temperature=opt_t)

confs_pre = np.max(probs_pre, axis=-1)
confs_post = np.max(probs_post, axis=-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(confs_pre, bins=10, alpha=0.7, color="crimson", edgecolor="black")
axes[0].set_title(f"Confidence Histogram (Pre-Cal, T=1.0)\nECE={scaler.ece_pre:.4f}")
axes[0].set_xlabel("Max Predicted Probability")
axes[0].set_ylabel("Frekuensi")

axes[1].hist(confs_post, bins=10, alpha=0.7, color="teal", edgecolor="black")
axes[1].set_title(f"Confidence Histogram (Post-Cal, T={opt_t:.2f})\nECE={scaler.ece_post:.4f}")
axes[1].set_xlabel("Max Predicted Probability")
axes[1].set_ylabel("Frekuensi")

plt.tight_layout()
plt.show()

---
## 6. Evaluasi Benchmark Resmi pada Test Set Holdout (495 Sampel)
Menguji Baseline vs FOMC-RoBERTa pada 495 sampel kalimat *unseen holdout* yang telah dibekukan sejak awal.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

test_texts = df_test["sentence"].tolist()
y_test_true = df_test["label"].tolist()

# 1. Prediksi Baseline
X_test_base = vec.transform(test_texts)
y_test_pred_base = clf.predict(X_test_base)

# 2. Prediksi FOMC-RoBERTa Terkalibrasi
roberta.calibrator = scaler
test_logits = roberta.predict_raw_logits(test_texts)
test_probs = roberta.calibrator.predict_proba(test_logits)
y_test_pred_roberta = np.argmax(test_probs, axis=-1)

# Confusion Matrices
cm_base = confusion_matrix(y_test_true, y_test_pred_base)
cm_roberta = confusion_matrix(y_test_true, y_test_pred_roberta)
labels_order = ["Dovish (0)", "Hawkish (1)", "Neutral (2)"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm_base, annot=True, fmt="d", cmap="Blues", xticklabels=labels_order, yticklabels=labels_order, ax=axes[0])
axes[0].set_title(f"Baseline TF-IDF + LR\nMacro F1: {f1_score(y_test_true, y_test_pred_base, average='macro'):.4f}")
axes[0].set_ylabel("True Stance")
axes[0].set_xlabel("Predicted Stance")

sns.heatmap(cm_roberta, annot=True, fmt="d", cmap="Greens", xticklabels=labels_order, yticklabels=labels_order, ax=axes[1])
axes[1].set_title(f"FOMC-RoBERTa (ACL 2023)\nMacro F1: {f1_score(y_test_true, y_test_pred_roberta, average='macro'):.4f}")
axes[1].set_ylabel("True Stance")
axes[1].set_xlabel("Predicted Stance")

plt.tight_layout()
plt.show()

---
## 7. Uji Keharusan Komponen (*Component Necessity & McNemar Test*)
Membuktikan secara ilmiah apakah keunggulan model Transformer signifikan secara statistik ($p < 0.05$) atau sekadar kebetulan.

In [ ]:
from model.fomc.evaluation.evaluate_holdout import compute_bootstrap_ci, compute_mcnemar_test

# 1. Uji McNemar
mcnemar_res = compute_mcnemar_test(y_test_true, y_test_pred_base, y_test_pred_roberta)
print("=== HASIL UJI MCNEMAR ===")
print(f"Tabel Kontingensi: {mcnemar_res['contingency_table']}")
print(f"Chi-Square Stat  : {mcnemar_res['chi2_statistic']:.4f}")
print(f"p-value          : {mcnemar_res['p_value']:.6f} (Signifikan p < 0.05: {mcnemar_res['significant_p_05']})")
print(f"Status Adopsi    : {'DITERIMA (Transformer Terbukti Perlu)' if mcnemar_res['component_adopted'] else 'DITOLAK'}")

# 2. Bootstrap 95% Confidence Interval untuk Delta F1
boot_res = compute_bootstrap_ci(y_test_true, y_test_pred_base, y_test_pred_roberta, n_bootstraps=1000, seed=SEED)
print("\n=== HASIL 1.000 BOOTSTRAP CI ===")
print(f"Mean Delta Macro F1 : +{boot_res['mean_delta_macro_f1']:.4f}")
print(f"95% Confidence Int  : [{boot_res['ci_95_lower']:.4f}, {boot_res['ci_95_upper']:.4f}]")

---
## 8. Analisis Kesalahan Kualitatif & Simulasi Transmisi Ganda
Memeriksa kalimat yang salah diprediksi dan memvalidasi transmisi polaritas berlawanan ke `RATE_UST` dan `SPX`.

In [ ]:
from model.fomc.src.data_loader import STANCE_TO_TONE

# Tampilkan 3 contoh kalimat test yang diprediksi salah untuk analisis kesalahan
error_indices = [i for i, (yt, yp) in enumerate(zip(y_test_true, y_test_pred_roberta)) if yt != yp]
print(f"Total kesalahan klasifikasi holdout: {len(error_indices)} dari {len(y_test_true)} kalimat\n")

for idx in error_indices[:3]:
    print(f"Kalimat : \"{test_texts[idx]}\"")
    print(f"Label Sebenarnya : {LABEL_TO_STANCE[y_test_true[idx]]}")
    print(f"Prediksi Model   : {LABEL_TO_STANCE[y_test_pred_roberta[idx]]}")
    print(f"Probabilitas     : Dovish={test_probs[idx][0]:.3f}, Hawkish={test_probs[idx][1]:.3f}, Neutral={test_probs[idx][2]:.3f}")
    print("-" * 80)

In [ ]:
# Demonstrasi Dual-Target Transmission ke Backend Schema
sample_sentences = [
    "Inflation pressures remain elevated, warranting further interest rate increases.",  # Hawkish
    "Economic growth is slowing noticeably, and policy easing may be required soon.",   # Dovish
    "The Committee decided to maintain the target range for the federal funds rate.",    # Neutral
]

for s in sample_sentences:
    art_sent = roberta.score_article("demo_id", s)
    prob = art_sent.probs[0]
    pred_class = np.argmax([prob.p_neg, prob.p_pos, prob.p_neu])
    stance = LABEL_TO_STANCE[pred_class]
    print(f"Teks    : \"{s}\"")
    print(f"Stance  : {stance.upper()} (Score: {art_sent.score:+.4f})")
    print(f"RATE_UST Tone : {STANCE_TO_TONE[(stance, 'RATE_UST')].upper()}")
    print(f"SPX Tone      : {STANCE_TO_TONE[(stance, 'SPX')].upper()}")
    print("=" * 80)

---
## 9. Ekspor Laporan Metrik Resmi & Auto-Zip Kaggle (Opsi B)
Menyimpan `benchmark_results.json` dan mengompres seluruh artefak keluaran jika dijalankan di cloud.

In [ ]:
from model.fomc.evaluation.evaluate_holdout import evaluate_holdout

# Simpan laporan benchmark resmi ke disk
benchmark_report = evaluate_holdout(skip_roberta_if_missing=True)
print("✅ Laporan benchmark resmi berhasil disimpan ke: model/fomc/evaluation/benchmark_results.json")

# FITUR KHUSUS OPSI B: Jika di Kaggle, zip seluruh artefak untuk unduh 1-klik
if IS_KAGGLE:
    !zip -q -r /kaggle/working/fomc_artifacts.zip model/fomc/weights/ model/fomc/evaluation/
    print("🎉 [KAGGLE] fomc_artifacts.zip berhasil dibuat di /kaggle/working/!")
    print("👉 Unduh file zip tersebut dari panel kanan Output Kaggle lalu ekstrak ke repo lokal laptop Anda.")
else:
    print("💻 [LOKAL] Seluruh artefak telah tersimpan langsung di folder lokal repositori Anda:")
    print("   - model/fomc/weights/baseline_fomc.joblib")
    print("   - model/fomc/weights/temperature.json")
    print("   - model/fomc/evaluation/benchmark_results.json")